# 04 Counts and yes/no outcomes: logistic, Poisson and negative binomial models, tables of counts, risks

simplydata's words for outcomes that are yes or no, or counts, checked on
- **birthwt** (Hosmer & Lemeshow 1989): low birth weight (yes or no) from the mother's age, weight, race and smoking;
- **NMES 1988** (Deb & Trivedi 1997; analysed by Zeileis, Kleiber & Jackman 2008): physician office visits of 4406
  people aged 66 or more (counts);
- the 2 x 2 table of smoking against low birth weight (birthwt).

**Only simplydata runs in this notebook.** The reference values are in `references/04_counts_and_yes_no.csv`, written
by `scripts/references/04_counts_and_yes_no.py`. First the printed ones: the Poisson and negative binomial fits of
the NMES data that Zeileis, Kleiber & Jackman (2008) printed (half a unit of the last printed digit) and the two risks
of the 2 x 2 table, exact by construction. Only where nothing is printed, a program is run once there: R `glm()` and
`MASS::glm.nb()`, run to tight convergence (`glm.control(epsilon = 1e-15)`), for the logistic model and the p-values
of the count models (relative difference 1e-6: each program stops its iterations at its own tolerance); R
`chisq.test`, `fisher.test`, `binom.test` and `prop.test`, SciPy for the G-test and statsmodels for the intervals of
the risk difference, risk ratio and odds ratio (closed formulas, 1e-10). Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, read_rows_by, versions

NOTEBOOK = "04_counts_and_yes_no"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def check_model(dataset, method, result, estimate_column):
    """Each term's estimate, standard error and p-value (the rows of the whole-column tests are left out)."""
    values = {}
    for row in read_rows(result):
        term = row["term"]
        if term.endswith("(whole column)"):
            continue
        values[f"{term} estimate"] = row[estimate_column]
        values[f"{term} standard error"] = row["standard error"]
        values[f"{term} p-value"] = row["p-value"]
    checks.check_values(dataset, method, values)

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Yes or no: `table.explain(outcome="yes or no")` (logistic regression)

The same underneath: statsmodels `GLM(family=Binomial())`; compared with R `glm(family = binomial)`. Race is a group
column; each group is compared with the first in A-Z order (black), as in R.

In [2]:
birthwt = load("birthwt")
found = birthwt.explain("low", using=["age", "lwt", "race", "smoke"], outcome="yes or no")

Explaining "low" from "age", "lwt", "race" and "smoke" (189 rows). Each odds ratio below holds the other columns fixed:
  intercept: 4.78, the odds of yes when every number column is 0 and every group column is at its baseline.
  age: each 1 more "age" multiplies the odds of yes by 0.978 (95 % CI 0.914 to 1.05); about 51 of 100 studies would show this much if there were truly none (p = 0.511).
  lwt: each 1 more "lwt" multiplies the odds of yes by 0.988 (95 % CI 0.975 to 1.000); about 5 of 100 studies would show this much if there were truly none (p = 0.050).
  race "other" against "black": the odds of yes is 0.749 times as high (95 % CI 0.267 to 2.10); about 58 of 100 studies would show this much if there were truly none (p = 0.584).
  race "white" against "black": the odds of yes is 0.292 times as high (95 % CI 0.106 to 0.804); about 2 of 100 studies would show this much if there were truly none (p = 0.017).
  smoke: each 1 more "smoke" multiplies the odds of yes by 2.87 (95 % CI 1.3

In [3]:
check_model("birthwt", "explain (yes or no)", found, "log odds ratio")

agrees  birthwt | explain (yes or no) | (intercept) estimate: simplydata 1.56412294503, R 1.56412294503; relative difference 3.83e-15 (allowed 1e-06)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/glm.html
agrees  birthwt | explain (yes or no) | (intercept) standard error: simplydata 1.16700522181, R 1.16700522181; relative difference 7.61e-16 (allowed 1e-06)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/glm.html
agrees  birthwt | explain (yes or no) | (intercept) p-value: simplydata 0.18015177297, R 0.18015177297; relative difference 7.09e-15 (allowed 1e-06)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/glm.html
agrees  birthwt | explain (yes or no) | age estimate: simplydata -0.0224782798746, R -0.0224782798746; relative difference 1.39e-15 (allowed 1e-06)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/glm.html
agrees  birthwt | explain

## Counts: `table.explain(outcome="counts")` (Poisson regression)

Office visits from hospital stays, self-rated health, chronic conditions, gender, years of school and private
insurance: the model of Zeileis, Kleiber & Jackman (2008), who printed its estimates and standard errors. The same
underneath: statsmodels `GLM(family=Poisson())`; the p-values against R `glm(family = poisson)`.

In [4]:
nmes = quiet(sdata.table, "../data/fetched/nmes1988.csv", separator=",")
USING = ["hospital", "health", "chronic", "gender", "school", "insurance"]
found = nmes.explain("visits", using=USING, outcome="counts")

Explaining "visits" from "hospital", "health", "chronic", "gender", "school" and "insurance" (4406 rows). Each rate ratio below holds the other columns fixed:
  intercept: 2.80, the expected count when every number column is 0 and every group column is at its baseline.
  hospital: each 1 more "hospital" multiplies the expected count by 1.18 (95 % CI 1.17 to 1.19); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  health "excellent" against "average": the expected count is 0.696 times as high (95 % CI 0.656 to 0.739); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  health "poor" against "average": the expected count is 1.28 times as high (95 % CI 1.24 to 1.33); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  chronic: each 1 more "chronic" multiplies the expected count by 1.16 (95 % CI 1.15 to 1.17); fewer than 1 in 1000 studies would show this much if there were

In [5]:
check_model("NMES 1988", "explain (counts)", found, "log rate ratio")

agrees  NMES 1988 | explain (counts) | (intercept) estimate: simplydata 1.02887419508, Zeileis, Kleiber & Jackman (2008), printed 1.028874; difference 1.95e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), printed (printed) - https://doi.org/10.18637/jss.v027.i08
agrees  NMES 1988 | explain (counts) | (intercept) standard error: simplydata 0.0237848912619, Zeileis, Kleiber & Jackman (2008), printed 0.023785; difference 1.09e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), printed (printed) - https://doi.org/10.18637/jss.v027.i08
agrees  NMES 1988 | explain (counts) | (intercept) p-value: simplydata 0, R 0; relative difference 0.00e+00 (allowed 1e-06)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/glm.html
agrees  NMES 1988 | explain (counts) | hospital estimate: simplydata 0.164797389209, Zeileis, Kleiber & Jackman (2008), printed 0.164797; difference 3.89e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Ja

## Counts that vary a lot: `table.explain(outcome="counts that vary a lot")` (negative binomial)

As R `MASS::glm.nb()`, which Zeileis et al. (2008) used (their Table 2 prints its estimates and standard errors to 3
decimals): theta by maximum likelihood, then the GLM with that theta held for the standard errors. The same
underneath: statsmodels `NegativeBinomial` (NB2) for theta, then `GLM(family=NegativeBinomial(alpha))`.

In [6]:
found = quiet(nmes.explain, "visits", using=USING, outcome="counts that vary a lot")
check_model("NMES 1988", "explain (counts that vary a lot)", found, "log rate ratio")

agrees  NMES 1988 | explain (counts that vary a lot) | (intercept) estimate: simplydata 0.929256614463, Zeileis, Kleiber & Jackman (2008), printed 0.929; difference 2.57e-04 (allowed 5e-04)
      Source: Zeileis, Kleiber & Jackman (2008), printed (printed) - https://doi.org/10.18637/jss.v027.i08
agrees  NMES 1988 | explain (counts that vary a lot) | (intercept) standard error: simplydata 0.0545912711659, Zeileis, Kleiber & Jackman (2008), printed 0.055; difference 4.09e-04 (allowed 5e-04)
      Source: Zeileis, Kleiber & Jackman (2008), printed (printed) - https://doi.org/10.18637/jss.v027.i08
explained difference – reference (C5)  NMES 1988 | explain (counts that vary a lot) | (intercept) p-value: simplydata 5.63399403405e-65, R 5.63403015329e-65; relative difference 6.41e-06 (allowed 1e-06)
      why: NMES 1988, negative binomial: 4 p-values against R MASS::glm.nb (1e-6 to 1.5e-5 relative) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c-the-reference-progr

## A 2 x 2 table of counts: `table.compare_counts_of`

Smoking in pregnancy against low birth weight (birthwt). The chi-square test with Yates' correction (the default for
a 2 x 2 table, as R `chisq.test()`), Fisher's exact test (R `fisher.test()`), and the G-test (no correction, as R
`DescTools::GTest()`; compared with SciPy `chi2_contingency(lambda_="log-likelihood")`).

In [7]:
counts = load("counts")
counts.compare_counts_of("low", for_each_group_in="smoke")

Comparing how often each "low" turns up in each "smoke" (2 x 2 table, 189 rows): Cramer's V = 0.161 (95 % CI 0 to 0.304; 0 means no link, 1 a complete link).
If "low" and "smoke" were truly unrelated, differences this big or bigger would turn up in about 4 of 100 studies like this one (p = 0.040).
Test: chi-square test of independence with Yates' continuity correction (as R chisq.test() does for 2 x 2 tables) (chi-square = 4.24, df = 1), the usual test for counts.


,test,what it compares,statistic name,statistic,degrees of freedom,p-value,effect size name,effect size,effect size 95 % CI low,effect size 95 % CI high,rows,smallest expected count
1,chi-square test of independence,"how often each ""low"" turns up in each ""smoke""",chi-square,4.235929,1.0,0.039577,"strength of the link, 0 (none) to 1 (complete)...",0.161404,0.0,0.304099,189,23.100529


In [8]:
for name, test in (("chi-square", None), ("fisher", "fisher"), ("g", "g")):
    options = {"test": test} if test else {}
    row = read_row(quiet(counts.compare_counts_of, "low", for_each_group_in="smoke", **options))
    checks.check_values("birthwt", f"compare_counts_of ({name})", {"statistic": row["statistic"], "p-value": row["p-value"]})

agrees  birthwt | compare_counts_of (chi-square) | statistic: simplydata 4.2359285456, R chisq.test 4.2359285456; relative difference 2.10e-16 (allowed 1e-10)
      Source: R chisq.test (run reference) - https://search.r-project.org/R/refmans/stats/html/chisq.test.html
agrees  birthwt | compare_counts_of (chi-square) | p-value: simplydata 0.0395769693252, R chisq.test 0.0395769693252; relative difference 7.01e-16 (allowed 1e-10)
      Source: R chisq.test (run reference) - https://search.r-project.org/R/refmans/stats/html/chisq.test.html
agrees  birthwt | compare_counts_of (fisher) | p-value: simplydata 0.0361764986912, R fisher.test 0.0361764986912; relative difference 2.88e-15 (allowed 1e-10)
      Source: R fisher.test (run reference) - https://search.r-project.org/R/refmans/stats/html/fisher.test.html


agrees  birthwt | compare_counts_of (g) | statistic: simplydata 4.86739665481, SciPy chi2_contingency(correction=False, lambda_='log-likelihood') 4.86739665481; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy chi2_contingency(correction=False, lambda_='log-likelihood') (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.chi2_contingency.html
agrees  birthwt | compare_counts_of (g) | p-value: simplydata 0.0273687587229, SciPy chi2_contingency(correction=False, lambda_='log-likelihood') 0.0273687587229; relative difference 1.90e-15 (allowed 1e-10)
      Source: SciPy chi2_contingency(correction=False, lambda_='log-likelihood') (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.chi2_contingency.html


## Risks: `table.compare_risks`

The risk of low birth weight for smokers against non-smokers: the risk difference with Newcombe's score interval,
the risk ratio and the odds ratio with Wald intervals on the log scale (statsmodels `confint_proportions_2indep`,
`Table2x2`), each group's risk with its exact (Clopper-Pearson) interval (R `binom.test`), and the p-value of the
chi-square test with Yates' correction (R `prop.test`). Hand count: 30 of 74 smokers, 29 of 115 non-smokers.

In [9]:
found = counts.compare_risks("low", for_each_group_in="smoke", against="non-smoker", value="yes")

"low" = "yes" (chosen with value=): 30 of 74 (40.5 %) in smoker and 29 of 115 (25.2 %) in non-smoker.
Out of 100, about 15 more in smoker than in non-smoker (risk difference 15.3 percentage points, 95 % CI 1.76 to 28.7).
Risk ratio 1.61 (95 % CI 1.06 to 2.44): the risk in smoker is 1.61 times the risk in non-smoker. Odds ratio 2.02 (95 % CI 1.08 to 3.78); odds are not risks: with an outcome this common (over 10 %) the odds ratio lies further from 1 than the risk ratio.
Number needed for one event more: 7 (1 / 0.153, rounded up; 95 % CI 3.48 to 56.9): for every 7 in smoker instead of non-smoker, one event more (the NNTH, if the event is harmful, such as death).
If the risks were truly the same, a difference this big or bigger would turn up in about 4 of 100 studies like this one (p = 0.040).
Method: the chi-square test with Yates' correction (SciPy, as R prop.test()) for the p-value; exact intervals for each risk, Newcombe's score interval for the difference (as R DescTools::BinomDiffCI

In [10]:
rows = read_rows(found)
by_measure = read_rows_by(found, "measure")
values = {}
for measure in ("risk difference", "risk ratio", "odds ratio"):
    values[f"{measure} estimate"] = by_measure[measure]["estimate"]
    values[f"{measure} 95 % CI low"] = by_measure[measure]["estimate 95 % CI low"]
    values[f"{measure} 95 % CI high"] = by_measure[measure]["estimate 95 % CI high"]
risks = [row for row in rows if row["measure"] == "risk"]          # the smokers first, then the non-smokers
for label, row in zip(("smoker", "non-smoker"), risks):
    values[f"{label} risk"] = row["estimate"]
    values[f"{label} risk 95 % CI low"] = row["estimate 95 % CI low"]
    values[f"{label} risk 95 % CI high"] = row["estimate 95 % CI high"]
values["p-value"] = by_measure["risk ratio"]["p-value"]
checks.check_values("birthwt", "compare_risks", values)

agrees  birthwt | compare_risks | risk difference estimate: simplydata 0.153231492362, statsmodels confint_proportions_2indep(newcomb) 0.153231492362; relative difference 5.43e-16 (allowed 1e-10)
      Source: statsmodels confint_proportions_2indep(newcomb) (run reference) - https://www.statsmodels.org/v0.15.0/generated/statsmodels.stats.proportion.confint_proportions_2indep.html
agrees  birthwt | compare_risks | risk difference 95 % CI low: simplydata 0.0175783320535, statsmodels confint_proportions_2indep(newcomb) 0.0175783320535; relative difference 7.70e-15 (allowed 1e-10)
      Source: statsmodels confint_proportions_2indep(newcomb) (run reference) - https://www.statsmodels.org/v0.15.0/generated/statsmodels.stats.proportion.confint_proportions_2indep.html
agrees  birthwt | compare_risks | risk difference 95 % CI high: simplydata 0.287117055494, statsmodels confint_proportions_2indep(newcomb) 0.287117055494; relative difference 3.87e-16 (allowed 1e-10)
      Source: statsmodels con

16

## Result

In [11]:
checks.finish()

04_counts_and_yes_no: 87 comparisons - 83 agrees, 4 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,04_counts_and_yes_no,birthwt,explain (yes or no),(intercept) estimate,1.564123,1.564123,R,-5.995204e-15,1.000000e-06,relative,3.832950e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,04_counts_and_yes_no,birthwt,explain (yes or no),(intercept) standard error,1.167005,1.167005,R,-8.881784e-16,1.000000e-06,relative,7.610749e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,04_counts_and_yes_no,birthwt,explain (yes or no),(intercept) p-value,0.180152,0.180152,R,1.276756e-15,1.000000e-06,relative,7.087116e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,04_counts_and_yes_no,birthwt,explain (yes or no),age estimate,-0.022478,-0.022478,R,-3.122502e-17,1.000000e-06,relative,1.389120e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,04_counts_and_yes_no,birthwt,explain (yes or no),age standard error,0.034170,0.034170,R,4.857226e-17,1.000000e-06,relative,1.421468e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,04_counts_and_yes_no,birthwt,compare_risks,smoker risk 95 % CI high,0.525898,0.525898,R binom.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
83,04_counts_and_yes_no,birthwt,compare_risks,non-smoker risk,0.252174,0.252174,exact by construction (29 of 115),5.551115e-17,1.000000e-10,relative,2.201304e-16,PASS,,agrees,,,,printed,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
84,04_counts_and_yes_no,birthwt,compare_risks,non-smoker risk 95 % CI low,0.175842,0.175842,R binom.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
85,04_counts_and_yes_no,birthwt,compare_risks,non-smoker risk 95 % CI high,0.341686,0.341686,R binom.test,1.665335e-16,1.000000e-10,relative,4.873880e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
